# 08a: Debug Notebook - Scalar Operations (L1)

## Purpose
Validate DSL scalar operations at Level 1 (event-level scalars).

## Phase 13.6.G Debug Notebook

This notebook tests:
- Scalar arithmetic (`event_weight * 2`)
- Scalar functions (`sqrt`, `abs`, `log`)
- Scalar comparisons (`event_weight > 1.0`)
- Type casting (`int(event_weight)`)

## Setup

In [ ]:
# Path setup - ensure RDataFrameDSL is importable
import sys
import os

# Add parent directory to path if running from examples/
notebook_dir = os.path.dirname(os.path.abspath('.'))
if 'RDataFrameDSL' not in sys.modules:
    # Try common locations
    for path in ['.', '..', notebook_dir]:
        if os.path.exists(os.path.join(path, 'RDataFrameDSL')):
            sys.path.insert(0, os.path.abspath(path))
            break

In [ ]:
import ROOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import time

from RDataFrameDSL import DSLCompiler
from RDataFrameDSL.verbosity import VERBOSE_DEFAULT, VERBOSE_FULL

# Configure logging for debug output
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("RDataFrameDSL")
# Uncomment for detailed debug output:
# logger.setLevel(logging.DEBUG)

# Configure plot size
plt.rcParams['figure.figsize'] = (6, 4)

print("Setup complete")

## Generate Test Data

Create simple test data with scalar columns for L1 testing.

In [ ]:
from tests.generators.toy_nd import generate_nd_2d_root

# Generate test file with scalar columns
filename = generate_nd_2d_root(size='S', seed=42)
rdf = ROOT.RDataFrame("Events", filename)

# Check available columns
print("Available columns:", list(rdf.GetColumnNames()))
print(f"Number of events: {rdf.Count().GetValue()}")

## Section 1: Scalar Arithmetic

### Description
Test basic arithmetic operations on scalar columns.

### What we're testing
- Multiplication: `event_weight * 2`
- Addition: `event_weight + 1`
- Division: `event_weight / 2`
- Combined: `(event_weight + 1) * 2`

### Expected behavior
DSL should produce identical results to direct NumPy computation.

In [ ]:
# Create DSL compiler
schema = {'event_id': 'long', 'event_weight': 'double', 'n_tracks': 'int'}
dsl = DSLCompiler(schema)

# Define scalar operations
dsl.define("weight_x2", "event_weight * 2")
dsl.define("weight_plus1", "event_weight + 1")
dsl.define("weight_div2", "event_weight / 2")
dsl.define("weight_combined", "(event_weight + 1) * 2")

# Inspect structure
print(dsl.describe_structure(VERBOSE_FULL))

In [ ]:
# Execute and validate
df = dsl.to_pandas(rdf, ['event_weight', 'weight_x2', 'weight_plus1', 'weight_div2', 'weight_combined'])

# Validation
assert np.allclose(df['weight_x2'], df['event_weight'] * 2)
assert np.allclose(df['weight_plus1'], df['event_weight'] + 1)
assert np.allclose(df['weight_div2'], df['event_weight'] / 2)
assert np.allclose(df['weight_combined'], (df['event_weight'] + 1) * 2)

print("✓ Scalar arithmetic: All validations passed")
df.head()

## Section 2: Scalar Functions

### Description
Test mathematical functions on scalar columns.

### What we're testing
- `sqrt(event_weight)`
- `abs(event_weight - 1.5)`
- `log(event_weight)`
- `exp(event_weight - 1)`

### If it fails
Check function mapping in `ir_builder.py` BUILTIN_FUNCTIONS.

In [ ]:
dsl2 = DSLCompiler(schema)

dsl2.define("sqrt_weight", "sqrt(event_weight)")
dsl2.define("abs_diff", "abs(event_weight - 1.5)")
dsl2.define("log_weight", "log(event_weight)")
dsl2.define("exp_shifted", "exp(event_weight - 1)")

df2 = dsl2.to_pandas(rdf, ['event_weight', 'sqrt_weight', 'abs_diff', 'log_weight', 'exp_shifted'])

# Validation
assert np.allclose(df2['sqrt_weight'], np.sqrt(df2['event_weight']))
assert np.allclose(df2['abs_diff'], np.abs(df2['event_weight'] - 1.5))
assert np.allclose(df2['log_weight'], np.log(df2['event_weight']))
assert np.allclose(df2['exp_shifted'], np.exp(df2['event_weight'] - 1))

print("✓ Scalar functions: All validations passed")
df2.head()

## Section 3: Scalar Comparisons

### Description
Test comparison operations returning boolean scalars.

### What we're testing
- Greater than: `event_weight > 1.5`
- Less than: `n_tracks < 5`
- Combined: `(event_weight > 1.0) & (n_tracks >= 3)`

In [ ]:
dsl3 = DSLCompiler(schema)

dsl3.define("high_weight", "event_weight > 1.5")
dsl3.define("few_tracks", "n_tracks < 5")
dsl3.define("combined", "(event_weight > 1.0) & (n_tracks >= 3)")

df3 = dsl3.to_pandas(rdf, ['event_weight', 'n_tracks', 'high_weight', 'few_tracks', 'combined'])

# Validation
assert np.array_equal(df3['high_weight'], df3['event_weight'] > 1.5)
assert np.array_equal(df3['few_tracks'], df3['n_tracks'] < 5)
expected_combined = (df3['event_weight'] > 1.0) & (df3['n_tracks'] >= 3)
assert np.array_equal(df3['combined'], expected_combined)

print("✓ Scalar comparisons: All validations passed")
print(f"  high_weight True count: {df3['high_weight'].sum()}")
print(f"  few_tracks True count: {df3['few_tracks'].sum()}")
print(f"  combined True count: {df3['combined'].sum()}")

## CPU Benchmarking

Compare DSL vs direct ROOT performance for scalar operations.

**Note:** This is diagnostic only, not a formal benchmark.

In [ ]:
def benchmark(name, func, n_runs=3):
    """Benchmark a function with CPU and wall time."""
    times_cpu = []
    times_wall = []
    
    for _ in range(n_runs):
        start_cpu = time.process_time()
        start_wall = time.perf_counter()
        result = func()
        times_cpu.append(time.process_time() - start_cpu)
        times_wall.append(time.perf_counter() - start_wall)
    
    print(f"{name}:")
    print(f"  CPU:  {np.mean(times_cpu):.4f}s ± {np.std(times_cpu):.4f}s")
    print(f"  Wall: {np.mean(times_wall):.4f}s ± {np.std(times_wall):.4f}s")
    return result, times_wall

# Benchmark: DSL scalar operation
dsl_bench = DSLCompiler(schema)
dsl_bench.define("scaled", "event_weight * 2")

_, times_dsl = benchmark("DSL (event_weight * 2)", 
                         lambda: dsl_bench.to_pandas(rdf, ['scaled']))

# Benchmark: Direct ROOT
_, times_root = benchmark("ROOT Direct",
                          lambda: rdf.AsNumpy(['event_weight']))

print(f"\nDSL overhead vs raw fetch: {np.mean(times_dsl)/np.mean(times_root):.2f}x")
print("(Note: DSL includes Define + flatten, ROOT is raw fetch only)")

## Summary

### Results

In [ ]:
print("=" * 50)
print("08a_debug_scalar.ipynb - L1 Scalar Operations")
print("=" * 50)
print("\n✓ Section 1: Scalar arithmetic - PASSED")
print("✓ Section 2: Scalar functions - PASSED")
print("✓ Section 3: Scalar comparisons - PASSED")
print("\n" + "=" * 50)
print("ALL TESTS PASSED")
print("=" * 50)